In [1]:
import pandas as pd
import os
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain_google_genai import ChatGoogleGenerativeAI
from itertools import cycle
import time
import re

c:\Users\Jim\anaconda3\envs\whisper-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
from dotenv import load_dotenv   # pip install python-dotenv
load_dotenv()
DEEPSEEK_API_KEY= os.getenv("DEEPSEEK_API_KEY")

In [3]:
from langchain_deepseek import ChatDeepSeek

MODEL_NAME = "deepseek-flash"  # or "deepseek-reasoner"

TEMPERATURE = 0

llm = ChatDeepSeek(
    model=MODEL_NAME,
    temperature=TEMPERATURE,
    api_key=DEEPSEEK_API_KEY
)

In [ ]:
template_positive_personal =  """
Rewrite the job description below as one flowing paragraph in plain prose — no headers, no bullet points, and no inline list-labels like "Responsibilities:" or "Requirements include:" followed by a semicolon list. Write connected sentences, not a bulleted list with the line breaks removed.

Keep everything specific to this role — responsibilities, required and preferred qualifications, tools/technologies, domain, seniority, experience, education, and any condition that affects whether someone qualifies — no matter which section it sits under. 

Job description:

{job_description}

**Your Reply:**
"""

In [5]:
cv_evaluator_prompt = """
You are an expert technical recruiter and resume screener. Your task is to analyze a candidate's CV against a provided Job Description and output a strictly valid JSON object detailing how well they match.

CRITICAL EVALUATION RULES:
1. Skills First: A "top match" requires strong alignment between the candidate's actual technical skills/tools and those requested in the job description.
2. Flexible Experience Threshold: Do not strictly penalize a candidate for falling slightly short on "years of experience" if their skills align. Specifically, if the candidate has around 2.5 to 3 years of experience (e.g., 2.8 years), you MUST consider their experience level a "top match" for roles asking for 0-3 years, and ALSO for roles asking for up to 4 years of experience.

You must output your analysis strictly as a valid JSON string matching the exact structure below. Do not include markdown formatting (like ```json), conversational text, or anything outside the curly braces.

JSON Structure:
{{
  "status": "[Select exactly one: 'top match', 'medium match', or 'low match'. Use 'top match' if the skills align well and experience fits within the flexible threshold above. Use 'medium match' if some core skills are present but major domain experience is missing. Use 'low match' if the candidate is in the wrong field or lacks the majority of mandatory skills.]",
  "matching_experience": "[Provide a clear, readable paragraph or summary detailing the exact skills, tools, and roles from the CV that directly satisfy the job description.]",
  "not_matching_experience": "[Provide a clear summary of the specific requirements, tools, or qualifications from the job description that are completely missing from the CV. If everything matches perfectly, state 'None - candidate meets all major requirements.']"
}}

Job description:

{job_description}

Candidate CV:

{cv}
"""



In [6]:
job_description = '''
Start of main content
Job title, keywords or company
City, county, postcode or 'remote'

Search
Wiley logo
Wiley logo
Senior Data Scientist (NLP + Applied AI)
Wiley
·
3.7
Remote
Remote
£44,200 - £63,400 a year
 -  
Full-time
Apply on company site



Job details navigation
Job
Job
1 of 2
Company
Company
2 of 2
Job details
Here’s how the job details align with your profile.
Pay

£44,200 - £63,400 a year
Job type

Full-time
Full job description
Job Description:


We believe in bold ideas, diverse perspectives, and the drive to transform knowledge into impact. Here, your curiosity fuels progress, your voice shapes innovation, and your ambition helps redefine what’s possible within science and learning. We are a culture that obsesses over impact, challenges, and drives what’s next to power infinite possibilities for our customers, colleagues and society at large.

About the Role:

About the role

We're building the systems that turn one of the world's largest scientific corpora into research intelligence. That means production NLP pipelines running over millions of journal articles, extracting entities, classifications, claim tuples, and summaries optimized for use by downstream agentic applications . We're looking for a senior data scientist to own domain-specific content modeling work end to end, from the eval set through the pipeline stage that ships it.

You'll join a small, senior team where data scientists own their models in production. You'll write the code, own the evaluations, ship the changes, and stay accountable for the outcomes. This is a hands-on role for someone who wants to see their models through to real users in a rapidly evolving market .

What you'll do

Design and build NLP enrichment pipelines that extract entities, classifications, claims, and summaries from scientific full-text at scale.

Compare NLP approaches to extraction and enrichment against LLM-based approaches, and pick the right tool for each task. That means putting traditional NLP (NER, sequence labeling, classification), embedding-based retrieval, LLM prompting, and fine-tuned smaller models on the same table, and defending each choice with evaluation, cost, and operational tradeoffs. This is a core part of the job, not an occasional exercise.

Own evaluation. Build the golden sets in consultation with SMEs and vendors, choose the metrics, and make productive tradeoffs between speed, quality, and cost.

Contribute to agentic AI application work: tool-using systems that reason over the enriched corpus, where your NLP and evaluation background will shape how the agent grounds and defends its answers.

Work directly with editors, product managers, and engineers. Bring the modeling perspective into product decisions, and translate stakeholder pushback into concrete modeling work.

What you'll bring

Strong NLP background across modern (LLMs, transformers, embeddings, retrieval) and classical (NER, classification, sequence labeling) approaches. You've built evaluations and learned from the results .

Clean python . You are comfortable in exploratory notebooks and production repositories, and an engineer taking over a modeling output from you has a good head start.

A habit of comparing approaches and choosing the right one for the task. You can defend "prompt a large LLM" and "train a small classifier on 2,000 labels" with equal seriousness, back the choice with an eval and a cost estimate, and know what to do when performance drifts.

Nice to have

Experience working with scientific or scholarly text.

Familiarity with AWS (S3, Batch, Lambda, SageMaker) and Parquet or Iceberg data lake patterns.

Experience running LLMs under real cost and latency budgets in production.

Some exposure to agentic AI applications: tool use, multi-step reasoning, guardrails, and evaluation of trajectories rather than single-turn outputs.

Why us

We publish some of the world's most-read research, and we're now in a rare position: applying modern AI to a corpus of trusted scientific knowledge that spans two centuries. Researchers will use the systems you build here to move faster and get closer to the answers they came for . That's the work : from knowledge to impact.


We power infinite possibilities.


For more than 200 years, we've transformed knowledge into discoveries that shape the world. Today, our global team of innovators, creators, and experts is driving what's next in science, education, and publishing—creating impact that reaches everywhere.

We're not just observers of progress. We're the ones accelerating scientific breakthroughs, advancing learning, and sparking innovation that redefines entire fields and improves lives.


Here, your talent matters. Your ideas have room to grow. And your work creates breakthroughs that can change everything.

Wiley is an equal opportunity/affirmative action employer. We evaluate all qualified applicants and treat all qualified applicants and employees without regard to race, color, religion, sex, sexual orientation, gender identity or expression, national origin, disability, protected veteran status, genetic information, or based on any individual's status in any group or class protected by applicable federal, state or local laws. Wiley is also committed to providing reasonable accommodation to applicants and employees with disabilities. Applicants who require accommodation to participate in the job application process may contact tasupport@wiley.com for assistance.


We are proud that our workplace promotes continual learning and internal mobility. We offer meeting-free Friday afternoons allowing more time for heads down work and professional development, and through a robust body of employee programing we facilitate a wide range of opportunities to foster community, learn, and grow.

We are committed to fair, transparent pay, and we strive to provide competitive compensation in addition to a comprehensive benefits package. The range below represents Wiley's good faith and reasonable estimate of the base pay for this role at the time of posting roles in the United Kingdom, Canada, USA, Austria, Czechia, Denmark, France, Greece, Italy, Netherlands, Romania, or Spain. It is anticipated that most qualified candidates will fall within the range, however the ultimate salary offered for this role may be higher or lower and will be set based on a variety of non-discriminatory factors, including but not limited to, geographic location, skills, and competencies.

When applying, please attach your resume/CV to be considered.

Salary Range:

44,200.00 GBP to 63,400.00 GBP&#xa;&#xa;#LI-CW1

Report job
Senior Data Scientist jobs in Remote
Jobs at Wiley in Remote
Senior Data Scientist salaries in Remote
Hiring Lab
Career advice
Browse jobs
Browse companies
Salaries
Indeed Events
Work at Indeed
Countries
About
Help
ESG at Indeed
© 2026 Indeed
Anti-Slavery statement
Accessibility at Indeed
Privacy Centre and Ad Choices
Terms'''

In [7]:
cv = '''MD RAWFUR MONZUR JIM
AI Engineer | Production GenAI/LLM Systems · Multi-Agent · NLP · Evaluation · Self-Hosted GPU Serving
47-49 East Parade, HG1 5LQ, Harrogate UK | 07765780804 | rawfurjim12@gmail.com |  LinkedIn URL 
PROFILE
AI Engineer with almost three years’ experience building production GenAI/LLM/NLP systems. Currently leading AI transformation at JudgeService Research Ltd, adding LLM features to a review-analytics platform and shipping new AI products: a multi-agent sentiment pipeline analysing 30,000 reviews/week and a paid review-response service used by 200+ UK dealerships (100k+ reviews/month, 93% drafts published unedited). Hands-on across the stack: Python, frontend/backend, prompt engineering, AI agents, multi-agent design, fine-tuning (LoRA/QLoRA), RAG, evaluation pipelines, self-hosted GPU serving, Docker, FastAPI, AWS/on-prem infrastructure, CI/CD, and dashboard-ready data visualization. Product-oriented, independent problem-solver with strong written and verbal communication skills, focused on reliable, production-grade AI.
KEY SKILLS
GenAI & LLM Engineering:  GenAI, Large Language Models (LLMs), NLP, prompt engineering, few-shot prompting, structured JSON output (Pydantic), RAG, LLM-as-a-judge, hallucination guards, evaluation-set design
Multi-Agent & Evaluation Pipelines:  multi-agent pipelines, AI agents, agentic workflows (Claude Code, MCP), vibe coding / AI-assisted development, evaluation pipelines, regression testing, precision/recall thresholds
Fine-Tuning & Models:  LoRA, QLoRA, Unsloth, knowledge distillation, Llama 3 (8B/70B), Gemma 4 31B, Gemini API, Groq
Backend, Infrastructure & CI:  Python, FastAPI, REST APIs, Docker, HAProxy, AWS EC2, NVIDIA GPU servers (cloud and on-prem), Ollama, GitHub Actions CI/CD, health checks and alerting
Data, Retrieval & Visualization:  LangChain, Chroma, embeddings (Linq-Embed-Mistral), WhisperX speech-to-text, SQL, star-schema data warehousing, Power BI, dashboard-ready JSON, data visualization
Programming & Engineering Practice:  JavaScript, TypeScript, frontend/backend development, OOP, PyTorch, scikit-learn, Git, unit and regression testing, record/replay LLM test harness, test coverage, production-grade code, Jira, stakeholder management, mentoring, end-to-end ownership
PROFESSIONAL EXPERIENCE
AI ENGINEER — GENAI / LLM SYSTEMS — JUDGESERVICE RESEARCH LTD, UK	Jan 2024 – Present
Multi-Agent Sentiment Analysis Pipeline
•	Designed and built a multi-agent LLM pipeline that turns raw customer reviews into structured JSON for two client-facing insight dashboards. Six single-purpose agents classify, analyse, extract evidence, link sentiment to named staff and verify overall sentiment. It analyses 30,000 reviews a week (1.5M+ a year) and contributed to a 20% lift in client retention.
•	Made outputs production-grade and auditable with JSON extraction, fuzzy matching to taxonomy labels, fixed schemas, loop guards and evidence grounding. Built a two-tier evaluation and regression suite that runs unit tests plus a hand-labelled hard-case set and fails CI if precision or recall drops. Raised subcategory recall from 0.78 to 0.96 (F1 0.98).
•	Cut inference costs with two-tier routing: latency-sensitive single reviews go to the Gemini API, while high-volume batch jobs run on a self-hosted Gemma 4 31B on company GPU hardware. Consolidated multiple models onto one to free over 10GB VRAM and let both products run concurrently.
•	Engineered zero-downtime model serving with HAProxy and active-passive Ollama servers, including health checks, automated failover, self-healing restarts and tiered alerting. Tuned concurrency and KV-cache reuse to roughly double throughput for the latency-critical endpoint.
AI Review-Response System
•	Built and shipped an LLM service that drafts customer-facing replies to car dealership reviews. It classifies each review as positive or negative, then generates three reply options (detailed, professional, short) following each dealership's own rules. Now handles 100k+ reviews a month, with 93% of drafts published without edits, and is live across major UK dealer groups.
•	Solved a hard generation constraint by fine-tuning Llama 3 8B with LoRA/QLoRA (Unsloth) on examples generated by Llama 3 70B. This knowledge-distillation approach let the small model inherit reliability on an exact brand-name rule while keeping 8B speed and low production cost.
•	Containerised the service with Docker behind a FastAPI REST API. Deployed first on AWS EC2, configuring NVIDIA drivers and the Linux environment from scratch, then led the move to the company's own GPU server to cut running costs. Added database caching so repeat reviews never trigger a second model call.
•	Added automatic quality gates (rule-based checks plus an LLM-as-a-judge step) so poor drafts never reach a human. Dealers pick one of three options every time, so their choices double as free feedback on which reply style works. The system has brought in roughly £50k in recurring revenue.
Local RAG Code-Documentation Engine
•	Built a RAG pipeline that automatically documents large legacy PHP and JavaScript codebases that previously had almost no documentation. Runs entirely on local hardware, so proprietary source code never leaves the company. Cut manual documentation effort by 90% and made onboarding new engineers far faster.
•	Designed retrieval end to end: LangChain ingestion, 2,000-character chunks, Linq-Embed-Mistral embeddings in a Chroma vector store. For each chunk the system retrieves related code from across the repository, so the model understands calls into other files before it writes. A local Llama 3 drafts the documentation, and a second technical-writer pass removes filler and formats clean Markdown.
•	Made it stable at scale: added a PyTorch GPU-memory safeguard (periodic cache clearing with short pauses) after early runs crashed on thousands of files, and refactored the pipeline into modular, object-oriented Python.
Call-Quality Assessment & AI Enablement
•	Built an automated call-quality assessment tool: WhisperX transcribes recorded calls and an LLM assesses each one against a set of quality rules, replacing manual QA audits.
•	Ran internal LLM and prompt-engineering workshops, speeding up adoption of AI tools across the team.
PERSONAL PROJECTS
 ResumeTailor (Built with Claude & Jira) [GitHub]
•	Engineered a six-agent pipeline (Gemini Flash) that analyzes job descriptions against a candidate’s real experience to precisely rewrite CV summaries and skills, boosting ATS without inventing missing skills.
•	Shipped this tested Python project in a single day using Claude Code to autonomously write product requirements, create 10 sequential Jira tickets via Atlassian MCP, and implement the code alongside 141 offline tests.
Government Spending Data Warehouse [GitHub]
•	Developed a data warehouse to analyze government spending patterns, facilitating data-driven decision-making for resource allocation.
•	Employed SQL for extracting, transforming, and loading data, pre-processing, and organizing it into a star schema platform for optimal query performance. Performed data analysis tasks and integrated the data with Power BI. 
•	Successfully pinpointed areas of high and low spending across various departments, enabling strategic budget adjustments and informed policy recommendations.
Personalised Diet Recommendation System [GitHub]
•	Leveraging Machine Learning, this system offers a Personalized Diet Recommendation System, fed by real-world data, with a robust CI/CD pipeline using GitHub Actions and Heroku for efficient integration, testing, and deployment. 
•	Used Docker for consistent functionality across environments, it uses the K-nearest Neighbors algorithm to tailor diet advice considering user metrics like age, height, weight, gender, activity level, BMI, and BMR.
•	Developed an interactive frontend using HTML, CSS, and JavaScript alongside a Flask web interface to facilitate user interaction, dynamically integrating API-fed data for precise, real-time nutritional advice to elevate the tool into a promoter of healthier lifestyle choices.
Multimodal Sarcasm Detection [GitHub]
•	Extracted images, comments and like counts from a screenshot dataset with OpenCV and EasyOCR.
•	Fused BERT and FastText text models with a ResNet image model to rate sarcasm and irony, working with the Facebook community Commenti Memorabili on their data.
Student Mathematics Performance Predictor [GitHub]
•	Designed a predictive model using machine learning algorithms and data science techniques, exhibiting exceptional problem-solving skills and innovation.
•	Utilized and Hyperparameter-tuned various algorithms and predict (RandomForest, DecisionTree, GradientBoosting, Linear Regression, XGBRegressor, CatBoosting, AdaBoost) for optimal results and showcasing my dedication in problem solving.
EDUCATION
MSc Data Science (Merit), London South Bank University, London, 2022
BSc Computer Science, East West University, Dhaka, Bangladesh, 2019
'''

In [8]:
# ==========================================
# OPTION 1: Modern LangChain Approach (LCEL)
# ==========================================
prompt_template = PromptTemplate.from_template(cv_evaluator_prompt)

# Create the chain using the pipe operator
chain = prompt_template | llm

# Execute the query passing the required variables
response = chain.invoke({
    "job_description": job_description,
    "cv": cv
})

# Get the raw string text (if using a ChatModel, use response.content)
json_output_string = response.content if hasattr(response, 'content') else response

In [9]:
json_output_string

'{"status":"top match","matching_experience":"The candidate has almost three years of hands-on AI engineering experience building production GenAI, LLM, and NLP systems, which fits the flexible experience threshold for this role. Their CV directly demonstrates Python, FastAPI, Docker, AWS EC2, GPU serving, CI/CD, RAG, embeddings, LangChain, Chroma, and production model deployment. They have built NLP pipelines for classification, evidence extraction, structured JSON generation, sentiment analysis, and multi-agent workflows at scale, including 30,000 reviews/week and 100k+ reviews/month. Their evaluation experience strongly matches the JD: hand-labelled hard-case sets, precision/recall thresholds, LLM-as-a-judge, regression testing in CI, and automatic quality gates. They also compare and select between LLM API calls, self-hosted smaller models, and fine-tuned LoRA/QLoRA models based on cost, latency, and reliability, and they have shipped models into production with measurable business

In [10]:
job_description = '''Data Engineer
Somerset Bridge Group
•
Newcastle upon Tyne NE1 4AD • Hybrid work
•
£53,000 - £67,000 a year
Apply now



Job details
Here’s how the job details align with your profile.
Pay

£53,000 - £67,000 a year
Job type

Permanent

Full-time
Benefits
Pulled from the full job description
Referral programme
Annual leave
Company pension
Cycle to work scheme
Car scheme
Full job description
Description

At Somerset Bridge Group, we're transforming our data platform to create a modern, cloud-based data ecosystem that powers smarter decision-making across the business. As part of this journey, we're consolidating our data capabilities onto a single Azure Databricks platform, leveraging technologies including Azure Data Factory, Delta Lake, Unity Catalog and Power BI to unlock new insights and drive innovation.
Joining our Enterprise Data Team, you'll play a key role in designing, building and optimising scalable data solutions that support Pricing, Underwriting, Claims and wider business operations. Working with large and complex datasets, you'll develop robust data pipelines and platforms that enable analytics, reporting, machine learning and AI-driven initiatives across the organisation.
You'll be responsible for building and maintaining data pipelines using Azure Databricks, Azure Data Factory and Delta Lake, supporting the modernisation of legacy platforms while ensuring data quality, reliability and performance. The role will also involve implementing data governance and security controls, optimising cloud-based data infrastructure, and supporting both real-time and batch data processing.

Working closely with Data Architects, Analysts, Pricing teams and Technology colleagues, you'll help shape the future of SBG's enterprise data platform, delivering scalable, secure and efficient solutions that support strategic business objectives and regulatory requirements.
This is an exciting opportunity to join a growing Enterprise Data Team and make a significant impact on the development of a modern Azure-based data platform, helping to drive data-led innovation across the business.

What you'll be responsible for:

Design, build and maintain scalable data pipelines using Azure Databricks, Azure Data Factory and Delta Lake to support both real-time and batch data processing.
Develop and optimise cloud-based data solutions within Azure, ensuring high performance, scalability and cost efficiency.
Create robust data models and lakehouse architectures to support analytics, reporting and machine learning initiatives.
Automate and monitor data workflows using tools such as Databricks Workflows, Delta Live Tables and Azure Monitor.
Ensure data quality, integrity and governance through effective controls, security standards and regulatory compliance.
Collaborate with stakeholders across Pricing, Underwriting, Data Science and Technology to deliver data-driven business solutions.
Implement data security and governance frameworks, including access controls, data lineage and sensitive data protection.
Drive continuous improvement by adopting new technologies and best practices across the Azure and Databricks ecosystem.

What you'll need:

Strong experience building data pipelines using Azure Data Factory, Databricks and Azure Data Lake.
Advanced SQL and Python (PySpark) skills for data transformation, automation and optimisation.
Hands-on experience with Delta Lake, Spark, Databricks Workflows and Structured Streaming.
Knowledge of data warehousing, lakehouse architectures and dimensional modelling.
Experience delivering scalable, high-quality data solutions to support analytics, reporting and machine learning.
Familiarity with real-time data processing, APIs and streaming technologies such as Kafka or Azure Event Hubs.
Experience with CI/CD, Azure DevOps, GitHub Actions and Infrastructure as Code.
Strong understanding of data governance, security and access controls, including Unity Catalog and Purview.
Experience monitoring, optimising and troubleshooting cloud-based data platforms.
Excellent problem-solving, communication and stakeholder management skills.
Collaborative approach with the ability to work effectively across technical and business teams.
Passion for innovation, continuous improvement and emerging data technologies.
Desirable
Experience with Scala.
Exposure to MLOps, feature stores and machine learning platforms.

Our Benefits

Hybrid working – 2 days in the office and 3 days working from home
25 days annual leave, rising to 27 days over 2 years’ service and 30 days after 5 years’ service. Plus bank holidays!
Discretionary annual bonus
Pension scheme – 5% employee, 6% employer
Flexible working – we will always consider applications for those who require less than the advertised hours
Flexi-time
Healthcare Cash Plan – claim cashback on a variety of everyday healthcare costs
Electric vehicle – salary sacrifice scheme
100’s of exclusive retailer discounts
Professional wellbeing, health & fitness app - Wrkit
Enhanced parental leave, including time off for IVF appointments
Religious bank holidays – if you don’t celebrate Christmas and Easter, you can use these annual leave days on other occasions throughout the year.
Life Assurance - 4 times your salary
25% Car Insurance Discount
20% Travel Insurance Discount
Cycle to Work Scheme
Employee Referral Scheme
Community support day

About Somerset Bridge Group
Somerset Bridge Group is dedicated to delivering fair products and innovative services in the insurance industry. Our group focuses on underwriting, broking, and claims handling to provide sustainable and innovative insurance solutions. Somerset Bridge Insurance Services Limited, operating under GoSkippy and Vavista, offers insurance coverage to over 700,000 customers. Somerset Bridge Limited handles underwriting and claims, processing almost 50,000 claims annually. Somerset Bridge Shared Services Limited provides essential support functions to ensure operational efficiency and compliance. With a strong commitment to values, culture, and customer service excellence, Somerset Bridge Group is recognised for its industry awards and growth. Join us to be part of a dynamic team that fosters creative thinking and personal development.
We are very proud to have been awarded a Gold Accreditation from Investors in People! We recognise that all of our people contribute to our success. That's why we are always looking for talented people to join our team - people who share our vision, who are passionate about what they do, and who want to be part of something special.
Equal Opportunity Employer
Somerset Bridge Group is committed to creating a diverse environment and is proud to be an Equal Opportunity Employer. We prohibit discrimination or harassment of any kind based on race, color, religion, national origin, sexual orientation, gender, gender identity or expression, age, pregnancy, physical or mental disability, genetic factors or other characteristics protected by law. SBG makes hiring decisions based solely on qualifications, skills and business requirements.'''

In [11]:
# ==========================================
# OPTION 1: Modern LangChain Approach (LCEL)
# ==========================================
prompt_template = PromptTemplate.from_template(cv_evaluator_prompt)

# Create the chain using the pipe operator
chain = prompt_template | llm

# Execute the query passing the required variables
response = chain.invoke({
    "job_description": job_description,
    "cv": cv
})

# Get the raw string text (if using a ChatModel, use response.content)
json_output_string = response.content if hasattr(response, 'content') else response

In [12]:
json_output_string

'{\n  "status": "low match",\n  "matching_experience": "The candidate has almost three years of relevant technical experience and demonstrates strength in Python, SQL, REST APIs, Docker, GitHub Actions CI/CD, data warehousing concepts, star-schema modelling, Power BI, and dashboard-ready data outputs. They have built production data-processing and AI pipelines, including multi-agent and RAG systems, and have experience with cloud infrastructure on AWS EC2, GPU serving, monitoring, health checks, failover, and data quality/evaluation controls. Their personal data warehouse project used SQL ETL, star schema design, and Power BI, which provides some transferable data engineering exposure. Communication, stakeholder management, and end-to-end ownership are also evidenced.",\n  "not_matching_experience": "The role is primarily an Azure Databricks Data Engineer position, and the candidate lacks the core mandatory Azure data engineering stack: Azure Data Factory, Azure Databricks, Delta Lake,

In [ ]:
res = chain.run(job_description=job_description) 

# 3. Print the result
print(res)